# Análisis exploratorio

Describe el corpus: reseñas por parque, distribución de estrellas e idioma.

## Rutas

Las rutas son relativas a la carpeta `notebooks/`. Los datos no forman parte del repositorio (ver README).

In [ ]:
# Comprobación de rutas
import os
PROY = os.path.abspath('..')
print("carpeta del notebook :", os.getcwd())
print("raiz del proyecto    :", PROY)
for _sub in ("Topic/Dataframes", "Topic/Resultados", "csv_parques"):
    _p = os.path.join(PROY, _sub)
    print(f"  {'OK ' if os.path.isdir(_p) else 'FALTA'}  {_sub}")
print("\nSi algo dice FALTA, el kernel no arranco en la carpeta del notebook,\n"
      "o los datos no estan donde se los espera (ver README).")

### Carga

In [ ]:
import pandas as pd
import matplotlib.ticker as mticker
from scipy import stats

df = pd.read_pickle("../df_generic_cleaned.pkl")
df.head()

### Reseñas por parque (Sección 3.4.1)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

#tabla frecuencia
tabla = (
    df.groupby('nombre_parque')
    .agg(cantidad_comentarios=('text', "count"))
    .reset_index()
    .sort_values("cantidad_comentarios", ascending=False)
)

total = tabla["cantidad_comentarios"].sum()
tabla["peso_relativo_%"] = (tabla["cantidad_comentarios"] / total * 100).round(2)

print(tabla.to_string(index=False))

# gráfico de barras
fig, ax = plt.subplots(figsize=(10, 6))

barras = ax.bar(
    tabla['nombre_parque'],
    tabla["cantidad_comentarios"],
    color="steelblue",
    edgecolor="white"
)

# poner etiqutas
for barra, (_, fila) in zip(barras, tabla.iterrows()):
    ax.text(
        barra.get_x() + barra.get_width() / 2,
        barra.get_height() + 0.3,
        f"{int(fila['cantidad_comentarios'])}\n({fila['peso_relativo_%']}%)",
        ha="center", va="bottom", fontsize=9
    )

ax.set_title("Cantidad de comentarios por parque", fontsize=14, fontweight="bold")
ax.set_xlabel("Parque")
ax.set_ylabel("Cantidad de comentarios")
ax.set_ylim(0, 7600)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig("EDA_comentarios_por_parque.png", dpi=300)
plt.show()
df.head()

### Distribución de estrellas (Figura 3.2, Sección 3.4.2)

In [ ]:
media    = df['stars'].mean()
mediana  = df['stars'].median()
sesgo    = df['stars'].skew()

# histograma de polaridad
fig1, ax1 = plt.subplots(figsize=(8, 6))
conteos = df['stars'].value_counts().sort_index()
bars = ax1.bar(conteos.index, conteos.values, color="steelblue", edgecolor="white", width=0.6)

for bar, val in zip(bars, conteos.values):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
             str(val), ha="center", va="bottom", fontsize=9)

ax1.axvline(media,   color="red",    linestyle="--", linewidth=1.5, label=f"Media: {media:.2f}")
ax1.axvline(mediana, color="orange", linestyle=":",  linewidth=1.5, label=f"Mediana: {mediana:.1f}")
ax1.set_title("Distribución de Polaridad")
ax1.set_xlabel("Estrellas")
ax1.set_ylabel("Cantidad de reseñas")
ax1.set_xticks([1, 2, 3, 4, 5])
ax1.legend(fontsize=8)
plt.tight_layout()
plt.savefig("../Topic/figuras/EDA_01_histograma_polaridad.png", dpi=300, bbox_inches="tight")
plt.show()

# boxplot de estrellas por parque
fig2, ax2 = plt.subplots(figsize=(10, 6))
parques       = df['nombre_parque'].unique()
datos_boxplot = [df[df['nombre_parque'] == p]['stars'].values for p in parques]

bp = ax2.boxplot(datos_boxplot, patch_artist=True, medianprops=dict(color="red", linewidth=2))
colores = plt.cm.tab10.colors
for patch, color in zip(bp['boxes'], colores):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax2.set_title("Distribución de Estrellas por Parque")
ax2.set_xlabel("Parque")
ax2.set_ylabel("Estrellas")
ax2.set_xticks(range(1, len(parques) + 1))
ax2.set_xticklabels(parques, rotation=30, ha="right", fontsize=8)
ax2.set_yticks([1, 2, 3, 4, 5])
plt.tight_layout()
plt.savefig("../Topic/figuras/EDA_02_boxplot_parques.png", dpi=300, bbox_inches="tight")
plt.show()

# barras aplidas por parque y estrellas
fig3, ax3 = plt.subplots(figsize=(10, 6))
tabla_apilada = (
    df.groupby(['nombre_parque', 'stars'])
    .size()
    .unstack(fill_value=0)
)
tabla_apilada_pct = tabla_apilada.div(tabla_apilada.sum(axis=1), axis=0) * 100

colores_estrellas = {1: "#d73027", 2: "#fc8d59", 3: "#fee090", 4: "#91bfdb", 5: "#4575b4"}
bottom = pd.Series([0.0] * len(tabla_apilada_pct), index=tabla_apilada_pct.index)

for estrella in sorted(tabla_apilada_pct.columns):
    valores = tabla_apilada_pct[estrella]
    ax3.bar(tabla_apilada_pct.index, valores,
            bottom=bottom,
            label=f"{estrella} ★",
            color=colores_estrellas.get(estrella, "grey"),
            edgecolor="white")
    bottom += valores

ax3.set_title("Composición de Estrellas por Parque (%)")
ax3.set_xlabel("Parque")
ax3.set_ylabel("Proporción (%)")
ax3.set_xticklabels(tabla_apilada_pct.index, rotation=30, ha="right", fontsize=8)
ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}%"))
ax3.legend(title="Estrellas", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig("../Topic/figuras/EDA_03_barras_apiladas_parques.png", dpi=300, bbox_inches="tight")
plt.show()

### Idioma (Sección 3.4.4)

In [ ]:
# pip install lingua-language-detector
from lingua import Language, LanguageDetectorBuilder

idiomas = [Language.SPANISH, Language.PORTUGUESE, Language.ENGLISH]
detector = LanguageDetectorBuilder.from_languages(*idiomas).build()

def detectar_idioma(texto, umbral=0.70):
    texto = (texto or "").strip()
    if len(texto) < 15:                       # texto muy corto no es confiable
        return "indeterminado"
    conf = detector.compute_language_confidence_values(texto)
    if not conf or conf[0].value < umbral:
        return "indeterminado"
    return conf[0].language.name.lower()

df["idioma"] = df["text"].apply(detectar_idioma)
dist = df["idioma"].value_counts(normalize=True).mul(100).round(2)
print(dist)